In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$ 
\mathbb{R} \times \mathbb{R} \to \mathbb{R}, \quad L(z, t) = 
\begin{cases} 0 & \text{if } z t > 0 \\ -z t & \text{if } z t \leq 0 \end{cases} 
$$

$$ 
\mathbb{R^n \times \mathbb{R^n}} \to \mathbb{R^n}, \quad L(z, t) = 
(L(z_1, t_1), L(z_2, t_2), \dots, L(z_n, t_n)) 
$$

$$ \text{Derivative} $$

$$ 
\frac{dL}{dz_i} = 
\begin{cases} 
    0 & \text{if } z_i t_i > 0 \\ 
    -t_i & \text{if } z_i t_i \leq 0 
\end{cases} 
$$

$$ \text{Jacobian} $$

$$
J_L(z) =
\begin{bmatrix}
    \frac{dL}{dz_1} & 0 & \cdots & 0 \\
    0 & \frac{dL}{dz_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{dL}{dz_n}
\end{bmatrix}
$$

$$ dL = J_L(z) \, dz $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dz}, dz \right \rangle =
\left \langle \frac{dF}{dL}, dL \right \rangle \\

\\[2em]

\left \langle \frac{dF}{dL}, dL \right \rangle = \\[0.5em]
\left \langle \frac{dF}{dL}, J_L(z) \, dz \right \rangle = \\[0.5em]
\left \langle {J_L(z)}^\top \, \frac{dF}{dL}, dz \right \rangle \implies \\[1em]

\frac{dF}{dz} = {J_L(z)}^\top \, \frac{dF}{dL} \\[0.5em]
\frac{dF}{dz} = \frac{dL}{dz} \odot \frac{dF}{dL}
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

def ploss(z, t, reduction = "mean"):
    """Calculate the `perceptron loss` between predicted `z` logits and true targets `t`."""

    if reduction != "mean":
        assert False, f"Unsupported reduction: {reduction}"

    assert tr.all((t == -1.0) | (t == +1.0))

    mask = (z * t <= 0)
    L = tr.zeros_like(z)
    L[mask] = - z[mask] * t[mask]

    return L.mean()


class PLossFunction(autograd.Function):
    """Custom autograd function for the `perceptron loss`."""

    @staticmethod
    def forward(ctx, z, t):
        ctx.save_for_backward(z, t)
        return ploss(z, t)

    @staticmethod
    def backward(ctx, dF_dL):
        (z, t) = ctx.saved_tensors
        S = z.shape[0]
        FO = z.shape[1]
        N = S * FO

        mask = (z * t <= 0)
        dy_dz = tr.zeros_like(z)
        dy_dz[mask] = -t[mask]
        dy_dz = 1 / N * dy_dz

        dF_dz = dy_dz * dF_dL

        return (dF_dz, None)


class PLoss(nn.Module):
    """Custom module for the `perceptron loss`."""

    def __init__(self, reduction: str = "mean"):
        super().__init__()

        if reduction != "mean":
            assert False, f"Unsupported reduction: {reduction}"

    def forward(self, z: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        return PLossFunction.apply(z, t)

In [ ]:
# Unit tests

def test_gradcheck(samples=10):
    def fn(z, t):
        return PLossFunction.apply(z, t)

    z = tr.randn(samples, 1, dtype=tr.float64, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0, dtype=tr.float64),
        tr.tensor(+1.0, dtype=tr.float64)
    )

    assert autograd.gradcheck(fn, (z, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    z = tr.randn(samples, 1, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0),
        tr.tensor(+1.0)
    ).reshape(samples, 1)

    z1 = z.clone().detach().requires_grad_(True)
    L1 = PLoss()(z1, t)
    L1.backward()

    z2 = z.clone().detach().requires_grad_(True)
    L2 = tr.max(tr.zeros_like(z2), - z2 * t).mean()
    L2.backward()

    assert L1.item() == approx(L2.item())
    assert z1.grad == approx(z2.grad)


if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)